In [6]:
import duckdb
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

con = duckdb.connect(r'data\gold.duckdb', read_only=True)
print('Connected to gold.duckdb')

def q(sql):
    return con.sql(sql).df()

Connected to gold.duckdb


In [7]:
# ── Change player name here to analyse anyone ────────────────────────────────
PLAYER = "V Kohli"
# ────────────────────────────────────────────────────────────────────────────
print(f"Showing IPL stats for: {PLAYER}")

Showing IPL stats for: V Kohli


## Batting

In [ ]:
# Career IPL batting summary
q(f"""
WITH bat AS (
    SELECT d.match_id, d.innings_number,
        sum(d.runs_batter)                                                              AS runs,
        sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END)                            AS balls,
        sum(CASE WHEN d.runs_batter = 4 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS fours,
        sum(CASE WHEN d.runs_batter = 6 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS sixes
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number
),
dis AS (
    SELECT DISTINCT d.match_id, d.innings_number
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.is_wicket AND d.wicket_player_out = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
),
all_inn AS (
    SELECT match_id, innings_number FROM bat
    UNION
    SELECT match_id, innings_number FROM dis
    UNION
    -- Player reached the crease as non-striker but innings ended before they faced a ball.
    -- Only included when the player does not bat anywhere else in the same match,
    -- which prevents Cricsheet data errors (wrong non_striker in opposing team's innings).
    SELECT DISTINCT d.match_id, d.innings_number
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.non_striker = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
      AND NOT EXISTS (
          SELECT 1 FROM main_marts.fct_deliveries d2
          WHERE d2.match_id = d.match_id AND d2.batter = '{PLAYER}'
      )
),
inn AS (
    SELECT
        a.match_id, a.innings_number,
        coalesce(b.runs,  0) AS runs,
        coalesce(b.balls, 0) AS balls,
        coalesce(b.fours, 0) AS fours,
        coalesce(b.sixes, 0) AS sixes,
        (d.match_id IS NOT NULL)::integer AS dismissed
    FROM all_inn a
    LEFT JOIN bat b USING (match_id, innings_number)
    LEFT JOIN dis d USING (match_id, innings_number)
)
SELECT
    count(DISTINCT match_id)                                    AS mat,
    count(*)                                                    AS inns,
    sum(1 - dismissed)                                         AS no,
    sum(runs)                                                   AS runs,
    max(runs)                                                   AS hs,
    round(sum(runs)::double / nullif(sum(dismissed), 0), 2)    AS ave,
    sum(balls)                                                  AS bf,
    round(sum(runs) * 100.0 / nullif(sum(balls), 0), 2)        AS sr,
    sum(CASE WHEN runs >= 100 THEN 1 ELSE 0 END)               AS "100",
    sum(CASE WHEN runs >= 50 AND runs < 100 THEN 1 ELSE 0 END) AS "50",
    sum(CASE WHEN runs = 0 AND dismissed = 1 THEN 1 ELSE 0 END) AS "0",
    sum(fours)                                                  AS "4s",
    sum(sixes)                                                  AS "6s"
FROM inn
""")

In [ ]:
# IPL batting by season
q(f"""
WITH bat AS (
    SELECT d.match_id, d.innings_number, m.season, d.batting_team,
        sum(d.runs_batter)                                                              AS runs,
        sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END)                            AS balls,
        sum(CASE WHEN d.runs_batter = 4 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS fours,
        sum(CASE WHEN d.runs_batter = 6 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS sixes
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, m.season, d.batting_team
),
dis AS (
    SELECT DISTINCT d.match_id, d.innings_number, m.season, d.batting_team
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.is_wicket AND d.wicket_player_out = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
),
all_inn AS (
    SELECT match_id, innings_number, season, batting_team FROM bat
    UNION
    SELECT match_id, innings_number, season, batting_team FROM dis
    UNION
    SELECT DISTINCT d.match_id, d.innings_number, m.season, d.batting_team
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.non_striker = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
      AND NOT EXISTS (
          SELECT 1 FROM main_marts.fct_deliveries d2
          WHERE d2.match_id = d.match_id AND d2.batter = '{PLAYER}'
      )
),
inn AS (
    SELECT
        a.match_id, a.innings_number, a.season, a.batting_team,
        coalesce(b.runs,  0) AS runs,
        coalesce(b.balls, 0) AS balls,
        coalesce(b.fours, 0) AS fours,
        coalesce(b.sixes, 0) AS sixes,
        (d.match_id IS NOT NULL)::integer AS dismissed
    FROM all_inn a
    LEFT JOIN bat b USING (match_id, innings_number)
    LEFT JOIN dis d USING (match_id, innings_number)
)
SELECT
    season,
    mode(batting_team)                                          AS team,
    count(DISTINCT match_id)                                    AS mat,
    count(*)                                                    AS inns,
    sum(1 - dismissed)                                         AS no,
    sum(runs)                                                   AS runs,
    max(runs)                                                   AS hs,
    round(sum(runs)::double / nullif(sum(dismissed), 0), 2)    AS ave,
    sum(balls)                                                  AS bf,
    round(sum(runs) * 100.0 / nullif(sum(balls), 0), 2)        AS sr,
    sum(CASE WHEN runs >= 100 THEN 1 ELSE 0 END)               AS "100",
    sum(CASE WHEN runs >= 50 AND runs < 100 THEN 1 ELSE 0 END) AS "50",
    sum(CASE WHEN runs = 0 AND dismissed = 1 THEN 1 ELSE 0 END) AS "0",
    sum(fours)                                                  AS "4s",
    sum(sixes)                                                  AS "6s"
FROM inn
GROUP BY season
ORDER BY season
""")

In [ ]:
# Top 20 highest IPL innings
q(f"""
WITH bat AS (
    SELECT d.match_id, d.innings_number, d.match_date, d.batting_team, d.bowling_team,
        m.season,
        sum(d.runs_batter)                                                              AS runs,
        sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END)                            AS balls,
        sum(CASE WHEN d.runs_batter = 4 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS fours,
        sum(CASE WHEN d.runs_batter = 6 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS sixes
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, d.match_date, d.batting_team, d.bowling_team, m.season
),
dis AS (
    SELECT DISTINCT d.match_id, d.innings_number
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.is_wicket AND d.wicket_player_out = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
)
SELECT
    b.season, b.match_date, b.batting_team, b.bowling_team AS vs,
    b.runs,
    (d.match_id IS NULL) AS not_out,
    b.balls,
    round(b.runs * 100.0 / nullif(b.balls, 0), 1) AS sr,
    b.fours, b.sixes
FROM bat b
LEFT JOIN dis d USING (match_id, innings_number)
ORDER BY b.runs DESC
LIMIT 20
""")

## Bowling

In [ ]:
# Career IPL bowling summary
q(f"""
WITH bowl AS (
    SELECT
        d.match_id,
        d.innings_number,
        sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0 THEN 1 ELSE 0 END)  AS legal_del,
        sum(d.runs_total - d.extras_byes - d.extras_legbyes)                           AS runs_conceded,
        sum(CASE WHEN d.is_wicket
                  AND d.wicket_kind NOT IN ('run out','retired hurt','retired out','obstructing the field')
             THEN 1 ELSE 0 END)                                                         AS wkts,
        sum(CASE WHEN d.extras_wides > 0 THEN 1 ELSE 0 END)                            AS wides,
        sum(CASE WHEN d.extras_noballs > 0 THEN 1 ELSE 0 END)                          AS no_balls
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.bowler = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number
),
maidens AS (
    SELECT d.match_id, d.innings_number, d.over_number,
        sum(d.runs_total - d.extras_byes - d.extras_legbyes) AS over_runs
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.bowler = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, d.over_number
    HAVING sum(d.runs_total - d.extras_byes - d.extras_legbyes) = 0
       AND count(*) >= 6
)
SELECT
    count(DISTINCT b.match_id)                                                          AS mat,
    count(*)                                                                            AS inns,
    (floor(sum(b.legal_del) / 6) || '.' || (sum(b.legal_del) % 6))::varchar            AS overs,
    (SELECT count(*) FROM maidens)                                                      AS mdns,
    sum(b.runs_conceded)                                                                AS runs,
    sum(b.wkts)                                                                         AS wkts,
    max_by(b.wkts || '/' || b.runs_conceded, b.wkts * 1000 - b.runs_conceded)          AS bbi,
    round(sum(b.runs_conceded)::double / nullif(sum(b.wkts), 0), 2)                    AS ave,
    round(sum(b.runs_conceded) * 6.0 / nullif(sum(b.legal_del), 0), 2)                 AS econ,
    round(sum(b.legal_del)::double / nullif(sum(b.wkts), 0), 2)                        AS sr,
    sum(CASE WHEN b.wkts >= 4 THEN 1 ELSE 0 END)                                       AS "4w",
    sum(CASE WHEN b.wkts >= 5 THEN 1 ELSE 0 END)                                       AS "5w"
FROM bowl b
""")

In [ ]:
# IPL bowling by season
q(f"""
WITH bowl AS (
    SELECT d.match_id, d.innings_number, m.season,
        sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0 THEN 1 ELSE 0 END)  AS legal_del,
        sum(d.runs_total - d.extras_byes - d.extras_legbyes)                           AS runs_conceded,
        sum(CASE WHEN d.is_wicket
                  AND d.wicket_kind NOT IN ('run out','retired hurt','retired out','obstructing the field')
             THEN 1 ELSE 0 END)                                                         AS wkts
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.bowler = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, m.season
)
SELECT
    season,
    count(DISTINCT match_id)                                                            AS mat,
    count(*)                                                                            AS inns,
    (floor(sum(legal_del) / 6) || '.' || (sum(legal_del) % 6))::varchar                AS overs,
    sum(runs_conceded)                                                                  AS runs,
    sum(wkts)                                                                           AS wkts,
    max_by(wkts || '/' || runs_conceded, wkts * 1000 - runs_conceded)                  AS bbi,
    round(sum(runs_conceded)::double / nullif(sum(wkts), 0), 2)                         AS ave,
    round(sum(runs_conceded) * 6.0 / nullif(sum(legal_del), 0), 2)                     AS econ,
    round(sum(legal_del)::double / nullif(sum(wkts), 0), 2)                             AS sr,
    sum(CASE WHEN wkts >= 4 THEN 1 ELSE 0 END)                                          AS "4w",
    sum(CASE WHEN wkts >= 5 THEN 1 ELSE 0 END)                                          AS "5w"
FROM bowl
GROUP BY season
ORDER BY season
""")

In [13]:
# Best bowling innings in IPL (top 20)
q(f"""
SELECT
    m.season,
    d.match_date,
    d.bowling_team,
    CASE WHEN m.team_1 = d.bowling_team THEN m.team_2 ELSE m.team_1 END AS vs,
    sum(CASE WHEN d.is_wicket
              AND d.wicket_kind NOT IN ('run out','retired hurt','retired out','obstructing the field')
         THEN 1 ELSE 0 END)                                              AS wkts,
    sum(d.runs_total - d.extras_byes - d.extras_legbyes)                 AS runs,
    (floor(sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0
                    THEN 1 ELSE 0 END) / 6)
     || '.' ||
     (sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0
               THEN 1 ELSE 0 END) % 6))::varchar                         AS overs
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m USING (match_id)
WHERE d.bowler = '{PLAYER}'
  AND m.event_name ILIKE '%indian premier league%'
  AND d.innings_number IN (1, 2)
GROUP BY m.season, d.match_id, d.innings_number, d.match_date, d.bowling_team, m.team_1, m.team_2
ORDER BY wkts DESC, runs
LIMIT 20
""")

,season,match_date,bowling_team,vs,wkts,runs,overs
0,2007/08,2008-05-25,Royal Challengers Bangalore,Sunrisers Hyderabad,2.0,25.0,3.0.4
1,2011,2011-05-22,Chennai Super Kings,Royal Challengers Bangalore,1.0,8.0,1.0.0
2,2011,2011-04-09,Royal Challengers Bangalore,Kochi Tuskers Kerala,1.0,14.0,2.0.0
3,2009/10,2010-03-14,Kolkata Knight Riders,Royal Challengers Bangalore,0.0,1.0,0.0.2
4,2015,2015-04-24,Royal Challengers Bangalore,Rajasthan Royals,0.0,4.0,0.0.5
5,2015,2015-05-04,Royal Challengers Bangalore,Chennai Super Kings,0.0,6.0,1.0.0
6,2011,2011-05-06,Royal Challengers Bangalore,Punjab Kings,0.0,7.0,1.0.0
7,2009,2009-04-20,Chennai Super Kings,Royal Challengers Bangalore,0.0,9.0,1.0.0
8,2011,2011-04-22,Royal Challengers Bangalore,Kolkata Knight Riders,0.0,9.0,1.0.0
9,2007/08,2008-05-12,Punjab Kings,Royal Challengers Bangalore,0.0,9.0,1.0.0


## Fielding

In [14]:
# Career IPL fielding summary
q(f"""
SELECT
    sum(CASE WHEN wicket_kind = 'caught'
              AND wicket_fielders LIKE '%{PLAYER}%' THEN 1 ELSE 0 END)          AS catches,
    sum(CASE WHEN wicket_kind = 'caught and bowled'
              AND bowler = '{PLAYER}' THEN 1 ELSE 0 END)                         AS caught_and_bowled,
    sum(CASE WHEN wicket_kind = 'stumped'
              AND wicket_fielders LIKE '%{PLAYER}%' THEN 1 ELSE 0 END)          AS stumpings,
    sum(CASE WHEN wicket_kind = 'run out'
              AND wicket_fielders LIKE '%{PLAYER}%' THEN 1 ELSE 0 END)          AS run_outs
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m USING (match_id)
WHERE m.event_name ILIKE '%indian premier league%'
  AND d.innings_number IN (1, 2)
  AND d.is_wicket
""")

,catches,caught_and_bowled,stumpings,run_outs
0,121.0,0.0,0.0,20.0


In [15]:
# IPL fielding by season
q(f"""
SELECT
    m.season,
    sum(CASE WHEN d.wicket_kind = 'caught'
              AND d.wicket_fielders LIKE '%{PLAYER}%' THEN 1 ELSE 0 END)        AS catches,
    sum(CASE WHEN d.wicket_kind = 'caught and bowled'
              AND d.bowler = '{PLAYER}' THEN 1 ELSE 0 END)                       AS caught_and_bowled,
    sum(CASE WHEN d.wicket_kind = 'stumped'
              AND d.wicket_fielders LIKE '%{PLAYER}%' THEN 1 ELSE 0 END)        AS stumpings,
    sum(CASE WHEN d.wicket_kind = 'run out'
              AND d.wicket_fielders LIKE '%{PLAYER}%' THEN 1 ELSE 0 END)        AS run_outs
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m USING (match_id)
WHERE m.event_name ILIKE '%indian premier league%'
  AND d.innings_number IN (1, 2)
  AND d.is_wicket
GROUP BY m.season
ORDER BY m.season
""")

,season,catches,caught_and_bowled,stumpings,run_outs
0,2007/08,2.0,0.0,0.0,0.0
1,2009,9.0,0.0,0.0,2.0
2,2009/10,3.0,0.0,0.0,2.0
3,2011,7.0,0.0,0.0,0.0
4,2012,7.0,0.0,0.0,2.0
5,2013,7.0,0.0,0.0,3.0
6,2014,7.0,0.0,0.0,1.0
7,2015,7.0,0.0,0.0,3.0
8,2016,6.0,0.0,0.0,1.0
9,2017,5.0,0.0,0.0,1.0


In [ ]:
# ── Diagnostic: check player name variations in IPL data ─────────────────────
# Searches for similar names in case Cricsheet uses a different spelling
q(f"""
SELECT DISTINCT batter AS player_name, 'batter' AS role
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m USING (match_id)
WHERE m.event_name ILIKE '%indian premier league%'
  AND (batter ILIKE '%rahul%' OR batter ILIKE '%kl%')
UNION ALL
SELECT DISTINCT wicket_player_out, 'dismissed'
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m USING (match_id)
WHERE m.event_name ILIKE '%indian premier league%'
  AND wicket_player_out ILIKE '%rahul%'
ORDER BY player_name
""")

In [ ]:
# ── Diagnostic: all IPL innings for PLAYER ordered by date ───────────────────
q(f"""
WITH bat AS (
    SELECT d.match_id, d.innings_number, d.match_date, d.batting_team, m.season,
        sum(d.runs_batter) AS runs,
        sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END) AS balls
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, d.match_date, d.batting_team, m.season
),
dis AS (
    SELECT DISTINCT d.match_id, d.innings_number, d.match_date, d.batting_team, m.season
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.is_wicket AND d.wicket_player_out = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
),
all_inn AS (
    SELECT match_id, innings_number, match_date, batting_team, season FROM bat
    UNION
    SELECT match_id, innings_number, match_date, batting_team, season FROM dis
    UNION
    SELECT DISTINCT d.match_id, d.innings_number, d.match_date, d.batting_team, m.season
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.non_striker = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
      AND NOT EXISTS (
          SELECT 1 FROM main_marts.fct_deliveries d2
          WHERE d2.match_id = d.match_id AND d2.batter = '{PLAYER}'
      )
)
SELECT
    a.season, a.match_date, a.batting_team,
    coalesce(b.runs,  0) AS runs,
    coalesce(b.balls, 0) AS balls,
    (d.match_id IS NOT NULL) AS dismissed
FROM all_inn a
LEFT JOIN bat b USING (match_id, innings_number)
LEFT JOIN dis d USING (match_id, innings_number)
ORDER BY a.match_date
""")